## import

In [10]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from pymongo import MongoClient
from datetime import datetime
import re
from time import sleep
import time

## connect

In [7]:
client = MongoClient('mongodb://localhost:27017/')
client.drop_database('stock')
db = client['stock']


KeyboardInterrupt: 

## collection

In [3]:
collection = db['MBB']

### empty 

In [9]:
all_page=[]
stocks=[]

## run webdriver

In [11]:
driver = webdriver.Chrome()
driver.get("https://simplize.vn/co-phieu/MBB/lich-su-gia")
sleep(10)
def crawl_data():
    body = driver.find_element(By.TAG_NAME, "body")
    for _ in range(10):  
        body.send_keys(Keys.END)
        sleep(2)
    # Lấy toàn bộ hàng trong bảng
    rows = driver.find_elements(By.CSS_SELECTOR, ".simplize-table-row.simplize-table-row-level-0")

    for row in rows:
        columns = row.find_elements(By.TAG_NAME, "td")
        _date = columns[0].text
        open_price = columns[1].text.replace(",", "")
        highest_price = columns[2].text.replace(",", "")
        lowest_price = columns[3].text.replace(",", "")
        closing_price = columns[4].text.replace(",", "")
        changed_price = columns[5].text.replace(",", "")
        if (changed_price=='-'):
            changed_price = 0
        price_change_percentage = columns[6].text.replace(",", "")
        if (price_change_percentage == '-'):
            price_change_percentage = 0
        changed_volume = columns[7].text.replace(",", "")
        data = {
            "date": _date,
            "open_price": open_price,
            "highest_price": highest_price,
            "lowest_price": lowest_price,
            "closing_price": closing_price,
            "changed_price": changed_price,
            "price_change_percentage": price_change_percentage,
            "changed_volume": changed_volume
        }
        stocks.append(data)
        
a = driver.find_element(By.XPATH , '//*[@id="phan-tich"]/div[2]/div/div/div[2]/div[1]/div/div[3]/ul/li[9]/div')
# print(len(a))
for i in range(1,4):
    print(i)
    crawl_data()
    page = a.click()
    # wait = WebDriverWait(driver, 20)
    # element = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="phan-tich"]/div[2]/div/div/div[2]/div[1]/div/div[3]/ul/li[9]/div')))
    # element.click()

1
2
3


In [12]:
print(len(stocks))

90


In [4]:
if stocks:
    try:
        collection.insert_many(stocks)
        print(f"Successfully insert {len(stocks)} stocks.")
    except Exception as e:
        print(f"Error insert stocks: {e}")

Error insert stocks: name 'collection' is not defined


In [ ]:
len(stocks)

30

## close drive

In [ ]:
driver.quit()

## query

In [ ]:
for demo in collection.find():
    print(demo)

In [ ]:
for demo in collection.find({'date': '17/10/2024'}):
    print(demo)

{'_id': ObjectId('6711338eebc9433f304582b5'), 'date': '17/10/2024', 'open_price': '25550', 'highest_price': '25900', 'lowest_price': '25350', 'closing_price': '25900', 'changed_price': '+400', 'price_change_percentage': '1.57%', 'changed_volume': '11681000'}


In [ ]:
for demo in collection.find().sort('highest_price',-1).limit(1):
    print(demo)

{'_id': ObjectId('6711338eebc9433f304582c1'), 'date': '01/10/2024', 'open_price': '25750', 'highest_price': '26150', 'lowest_price': '25650', 'closing_price': '25650', 'changed_price': '-50', 'price_change_percentage': '-0.19%', 'changed_volume': '18768300'}


## close DB

In [ ]:
# client.close()